---
## Question 1

> Explain the fundamental differences between DDL, DML, and DQL commands in SQL. Provide one example for each type of command.

### Answer

SQL statements are grouped into sub-languages according to what they do. **DDL** works on the structure of the database, **DML** works on the data stored inside that structure, and **DQL** is used only to read data.

| Feature | DDL | DML | DQL |
|---|---|---|---|
| Full form | Data Definition Language | Data Manipulation Language | Data Query Language |
| Purpose | Defines or changes the structure (schema) of database objects | Inserts, modifies or deletes the data (rows) stored in tables | Retrieves data from one or more tables |
| Works on | Databases, tables, indexes, views, etc. | Records / rows inside tables | Records / rows (read-only) |
| Commands | `CREATE`, `ALTER`, `DROP`, `TRUNCATE`, `RENAME` | `INSERT`, `UPDATE`, `DELETE` | `SELECT` |
| Effect on data | Changes the table definition; not row-by-row | Changes the data in the table | Does not change any data |
| Rollback | Auto-committed in MySQL, cannot be rolled back | Can be rolled back inside a transaction | Nothing to roll back |

**Examples**

**DDL** – create a table:

In [ ]:
%%sql
CREATE TABLE Students (
    StudentID INT PRIMARY KEY,
    StudentName VARCHAR(50)
);

**DML** – insert a record into the table:

In [ ]:
%%sql
INSERT INTO Students (StudentID, StudentName)
VALUES (1, 'Asha');

**DQL** – read the records from the table:

In [ ]:
%%sql
SELECT StudentID, StudentName
FROM Students;

Note: some textbooks treat `SELECT` as part of DML. It is shown separately as DQL here because it only reads data and never modifies it.

---
## Question 2

> What is the purpose of SQL constraints? Name and describe three common types of constraints, providing a simple scenario where each would be useful.

### Answer

**Purpose:** SQL constraints are rules applied to columns or tables that control what data may be stored. They enforce data integrity, accuracy and consistency by rejecting invalid data automatically at the database level (for example duplicate IDs, missing values, or references to records that do not exist), so the application does not have to check for these everywhere.

| Constraint | Description | Scenario where it is useful |
|---|---|---|
| PRIMARY KEY | Uniquely identifies each row in a table. Its values must be unique and cannot be NULL. A table can have only one primary key. | In a Students table, StudentID identifies each student, so no two students share an ID and no student is stored without one. |
| FOREIGN KEY | A column that refers to the primary key of another table, enforcing referential integrity between the two tables. | Orders.CustomerID must match an existing Customers.CustomerID, so an order cannot be created for a customer that does not exist. |
| UNIQUE | Ensures that all values in a column (or a set of columns) are different. NULLs are allowed (MySQL permits multiple NULLs). | In a Users table the Email column is UNIQUE, so two accounts cannot register with the same email address. |

Example showing all three constraints:

In [ ]:
%%sql
CREATE TABLE Customers (
    CustomerID INT PRIMARY KEY,      -- Primary key
    Email VARCHAR(100) UNIQUE        -- Unique
);

CREATE TABLE Orders (
    OrderID INT PRIMARY KEY,
    CustomerID INT,
    -- Foreign key
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);

Other common constraints are `NOT NULL` (a value is mandatory), `CHECK` (a value must satisfy a condition, e.g. Price > 0) and `DEFAULT` (a value used when none is supplied).

---
## Question 3

> Explain the difference between LIMIT and OFFSET clauses in SQL. How would you use them together to retrieve the third page of results, assuming each page has 10 records?

### Answer

- **LIMIT** sets the maximum number of rows that the query returns.
- **OFFSET** sets how many rows are skipped from the start of the result set before rows begin to be returned.

Used together they implement pagination. The offset for a page is calculated as:

`OFFSET = (page_number - 1) * page_size`

For page 3 with 10 records per page: OFFSET = (3 - 1) × 10 = **20**, and LIMIT = **10**. The first 20 rows (pages 1 and 2) are skipped and the next 10 rows (rows 21–30) are returned.

*(This example uses the `Products` table from Question 6, so run Question 6 first.)*

In [ ]:
%%sql
USE ECommerceDB;

SELECT *
FROM Products
ORDER BY ProductID
LIMIT 10 OFFSET 20;

MySQL also accepts the shorthand `LIMIT 20, 10` (offset first, then count). An `ORDER BY` clause should always be used with LIMIT/OFFSET so that the row order, and therefore the pages, are consistent.

*(With only 8 products in the table, this query returns an empty result; it demonstrates the syntax for page 3.)*

---
## Question 4

> What is a Common Table Expression (CTE) in SQL, and what are its main benefits? Provide a simple SQL example demonstrating its usage.

### Answer

A **Common Table Expression (CTE)** is a temporary, named result set defined with the `WITH` keyword. It exists only for the duration of the single statement (`SELECT`, `INSERT`, `UPDATE` or `DELETE`) that immediately follows it, and can be referred to like a table or view.

**Main benefits**

- **Readability:** complex queries are broken into named, logical steps instead of deeply nested subqueries.
- **Reusability:** the same CTE can be referenced several times in the main query.
- **Easier maintenance and debugging:** each block can be tested on its own.
- **Recursion:** recursive CTEs can query hierarchical data such as organisation charts or category trees.
- **No permanent object:** nothing is stored in the database, unlike a view or temporary table.

**Example** (using the ECommerceDB tables from Question 6)

Find customers whose total spending is more than 200:

In [ ]:
%%sql
USE ECommerceDB;

WITH CustomerSpend AS (
    SELECT CustomerID, SUM(TotalAmount) AS TotalSpent
    FROM Orders
    GROUP BY CustomerID
)
SELECT c.CustomerName, cs.TotalSpent
FROM Customers c
JOIN CustomerSpend cs ON c.CustomerID = cs.CustomerID
WHERE cs.TotalSpent > 200
ORDER BY cs.TotalSpent DESC;

**Expected output**

| CustomerName | TotalSpent |
|---|---|
| Alice Wonderland | 1410.50 |
| Bob the Builder | 219.99 |

---
## Question 5

> Describe the concept of SQL Normalization and its primary goals. Briefly explain the first three normal forms (1NF, 2NF, 3NF).

### Answer

**Normalization** is the process of organizing the tables and columns of a relational database so that data is divided into smaller, related tables and each fact is stored in only one place. It is applied step by step through a series of rules called normal forms.

**Primary goals**

- Reduce data redundancy (the same data stored repeatedly).
- Avoid insertion, update and deletion anomalies.
- Improve data integrity and consistency.
- Make the database easier to maintain and extend, and use storage efficiently.

**The first three normal forms**

| Normal form | Rule | Example of a violation | How to fix it |
|---|---|---|---|
| 1NF | Every column holds atomic (single) values, there are no repeating groups, and each row is uniquely identifiable. | A Phone column containing "98765, 91234" in one cell. | Store each phone number in its own row (or a separate Phone table). |
| 2NF | Must be in 1NF, and every non-key column depends on the whole primary key (no partial dependency). Relevant for composite keys. | OrderItems(OrderID, ProductID, ProductName, Qty): ProductName depends only on ProductID. | Move ProductName to a Products table keyed by ProductID. |
| 3NF | Must be in 2NF, and non-key columns depend only on the primary key, not on other non-key columns (no transitive dependency). | Employees(EmpID, DeptID, DeptName): DeptName depends on DeptID, not EmpID. | Move DeptName to a Departments table keyed by DeptID. |

---
## Question 6

> Create a database named ECommerceDB and perform the following tasks: (1) create the Categories, Products, Customers and Orders tables with appropriate data types and constraints; (2) insert the given records into each table.

### Answer

**Step 1: Create and select the database**

In [ ]:
%%sql
CREATE DATABASE ECommerceDB;
USE ECommerceDB;

**Step 2: Create the tables**

Parent tables are created first so that the foreign keys can reference them.

In [ ]:
%%sql
CREATE TABLE Categories (
    CategoryID   INT PRIMARY KEY,
    CategoryName VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE Products (
    ProductID     INT PRIMARY KEY,
    ProductName   VARCHAR(100) NOT NULL UNIQUE,
    CategoryID    INT,
    Price         DECIMAL(10,2) NOT NULL,
    StockQuantity INT,
    FOREIGN KEY (CategoryID) REFERENCES Categories(CategoryID)
);

CREATE TABLE Customers (
    CustomerID   INT PRIMARY KEY,
    CustomerName VARCHAR(100) NOT NULL,
    Email        VARCHAR(100) UNIQUE,
    JoinDate     DATE
);

CREATE TABLE Orders (
    OrderID     INT PRIMARY KEY,
    CustomerID  INT,
    OrderDate   DATE NOT NULL,
    TotalAmount DECIMAL(10,2),
    FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);

**Step 3: Insert the records**

In [ ]:
%%sql
-- Categories
INSERT INTO Categories (CategoryID, CategoryName) VALUES
(1, 'Electronics'),
(2, 'Books'),
(3, 'Home Goods'),
(4, 'Apparel');

-- Products
INSERT INTO Products (ProductID, ProductName, CategoryID, Price, StockQuantity) VALUES
(101, 'Laptop Pro',            1, 1200.00,  50),
(102, 'SQL Handbook',          2,   45.50, 200),
(103, 'Smart Speaker',         1,   99.99, 150),
(104, 'Coffee Maker',          3,   75.00,  80),
(105, 'Novel : The Great SQL', 2,   25.00, 120),
(106, 'Wireless Earbuds',      1,  150.00, 100),
(107, 'Blender X',             3,  120.00,  60),
(108, 'T-Shirt Casual',        4,   20.00, 300);

-- Customers
INSERT INTO Customers (CustomerID, CustomerName, Email, JoinDate) VALUES
(1, 'Alice Wonderland', 'alice@example.com',   '2023-01-10'),
(2, 'Bob the Builder',  'bob@example.com',     '2022-11-25'),
(3, 'Charlie Chaplin',  'charlie@example.com', '2023-03-01'),
(4, 'Diana Prince',     'diana@example.com',   '2021-04-26');

-- Orders
INSERT INTO Orders (OrderID, CustomerID, OrderDate, TotalAmount) VALUES
(1001, 1, '2023-04-26', 1245.50),
(1002, 2, '2023-10-12',   99.99),
(1003, 1, '2023-07-01',  145.00),
(1004, 3, '2023-01-14',  150.00),
(1005, 2, '2023-09-24',  120.00),
(1006, 1, '2023-06-19',   20.00);

Verify the data with `SELECT * FROM <table_name>;` for each table (Categories: 4 rows, Products: 8 rows, Customers: 4 rows, Orders: 6 rows).

In [ ]:
%%sql
SELECT * FROM Categories;

In [ ]:
%%sql
SELECT * FROM Products;

In [ ]:
%%sql
SELECT * FROM Customers;

In [ ]:
%%sql
SELECT * FROM Orders;

---
## Question 7

> Generate a report showing CustomerName, Email, and the TotalNumberofOrders for each customer. Include customers who have not placed any orders, in which case their TotalNumberofOrders should be 0. Order the results by CustomerName.

### Answer

In [ ]:
%%sql
SELECT c.CustomerName,
       c.Email,
       COUNT(o.OrderID) AS TotalNumberofOrders
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.Email
ORDER BY c.CustomerName;

**Explanation:** a `LEFT JOIN` keeps every customer even when there is no matching order. `COUNT(o.OrderID)` counts only non-NULL order IDs, so a customer with no orders (Diana Prince) gets 0.

**Expected output**

| CustomerName | Email | TotalNumberofOrders |
|---|---|---|
| Alice Wonderland | alice@example.com | 3 |
| Bob the Builder | bob@example.com | 2 |
| Charlie Chaplin | charlie@example.com | 1 |
| Diana Prince | diana@example.com | 0 |

---
## Question 8

> Retrieve Product Information with Category: Write a SQL query to display the ProductName, Price, StockQuantity, and CategoryName for all products. Order the results by CategoryName and then ProductName alphabetically.

### Answer

In [ ]:
%%sql
SELECT p.ProductName,
       p.Price,
       p.StockQuantity,
       c.CategoryName
FROM Products p
JOIN Categories c ON p.CategoryID = c.CategoryID
ORDER BY c.CategoryName, p.ProductName;

**Expected output**

| ProductName | Price | StockQuantity | CategoryName |
|---|---|---|---|
| T-Shirt Casual | 20.00 | 300 | Apparel |
| Novel : The Great SQL | 25.00 | 120 | Books |
| SQL Handbook | 45.50 | 200 | Books |
| Laptop Pro | 1200.00 | 50 | Electronics |
| Smart Speaker | 99.99 | 150 | Electronics |
| Wireless Earbuds | 150.00 | 100 | Electronics |
| Blender X | 120.00 | 60 | Home Goods |
| Coffee Maker | 75.00 | 80 | Home Goods |

---
## Question 9

> Write a SQL query that uses a Common Table Expression (CTE) and a Window Function (specifically ROW_NUMBER() or RANK()) to display the CategoryName, ProductName, and Price for the top 2 most expensive products in each CategoryName.

### Answer

In [ ]:
%%sql
WITH RankedProducts AS (
    SELECT c.CategoryName,
           p.ProductName,
           p.Price,
           ROW_NUMBER() OVER (
               PARTITION BY c.CategoryName
               ORDER BY p.Price DESC
           ) AS rn
    FROM Products p
    JOIN Categories c ON p.CategoryID = c.CategoryID
)
SELECT CategoryName, ProductName, Price
FROM RankedProducts
WHERE rn <= 2
ORDER BY CategoryName, Price DESC;

**Explanation:** the CTE numbers the products inside each category from the most expensive (rn = 1) to the cheapest using `ROW_NUMBER()` with `PARTITION BY CategoryName`. The outer query keeps only rn ≤ 2. Apparel has only one product, so only one row is returned for it. Window functions need MySQL 8.0 or later. `RANK()` can be used instead if products with equal prices should share a rank.

**Expected output**

| CategoryName | ProductName | Price |
|---|---|---|
| Apparel | T-Shirt Casual | 20.00 |
| Books | SQL Handbook | 45.50 |
| Books | Novel : The Great SQL | 25.00 |
| Electronics | Laptop Pro | 1200.00 |
| Electronics | Wireless Earbuds | 150.00 |
| Home Goods | Blender X | 120.00 |
| Home Goods | Coffee Maker | 75.00 |

---
## Question 10

> Sakila Video Rentals: using the Sakila database, answer the five business questions below.

### Answer

**Sakila tables used:** `customer`, `payment`, `rental`, `inventory`, `film`, `film_category`, `category`, `store`.

In [ ]:
%%sql
USE sakila;

### Task 1: Top 5 customers by total amount spent

In [ ]:
%%sql
SELECT c.customer_id,
       CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       c.email,
       SUM(p.amount) AS total_spent
FROM customer c
JOIN payment p ON c.customer_id = p.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.email
ORDER BY total_spent DESC
LIMIT 5;

Payments are summed for each customer, sorted from highest to lowest, and the first 5 rows are kept.

### Task 2: Three movie categories with the highest rental counts

In [ ]:
%%sql
SELECT cat.name AS category_name,
       COUNT(r.rental_id) AS rental_count
FROM category cat
JOIN film_category fc ON cat.category_id = fc.category_id
JOIN inventory i      ON fc.film_id = i.film_id
JOIN rental r         ON i.inventory_id = r.inventory_id
GROUP BY cat.category_id, cat.name
ORDER BY rental_count DESC
LIMIT 3;

The path is category → film_category → inventory → rental, so every rental is attributed to the category of the film that was rented.

### Task 3: Films available at each store and how many were never rented

In [ ]:
%%sql
WITH store_film AS (
    SELECT i.store_id,
           i.film_id,
           COUNT(r.rental_id) AS times_rented
    FROM inventory i
    LEFT JOIN rental r ON i.inventory_id = r.inventory_id
    GROUP BY i.store_id, i.film_id
)
SELECT store_id,
       COUNT(*) AS total_films_available,
       SUM(CASE WHEN times_rented = 0 THEN 1 ELSE 0 END)
           AS films_never_rented
FROM store_film
GROUP BY store_id;

The CTE gives one row per film per store with its rental count (a `LEFT JOIN` keeps films that were never rented, with a count of 0). The main query then counts the films for each store and how many of them have 0 rentals.

### Task 4: Total revenue per month for 2023

In [ ]:
%%sql
SELECT MONTH(payment_date)     AS month_number,
       MONTHNAME(payment_date) AS month_name,
       SUM(amount)             AS total_revenue
FROM payment
WHERE payment_date >= '2023-01-01'
  AND payment_date <  '2024-01-01'
GROUP BY MONTH(payment_date), MONTHNAME(payment_date)
ORDER BY month_number;

Note: the standard Sakila sample data contains payments only from 2005–2006, so this query returns no rows on the default database. To see output on the sample data, change the two dates to a year that exists (for example 2005-01-01 and 2006-01-01). On a database that has 2023 data, the query works as written.

### Task 5: Customers who rented more than 10 times in the last 6 months

In [ ]:
%%sql
SELECT c.customer_id,
       CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       c.email,
       COUNT(r.rental_id) AS rentals_last_6_months
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id
WHERE r.rental_date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH)
GROUP BY c.customer_id, c.first_name, c.last_name, c.email
HAVING COUNT(r.rental_id) > 10
ORDER BY rentals_last_6_months DESC;

Note: because the sample Sakila rentals are old, `CURDATE()` will return no rows. To demonstrate the logic on the sample data, measure the 6 months back from the latest rental instead by replacing the WHERE condition with:

`WHERE r.rental_date >= DATE_SUB((SELECT MAX(rental_date) FROM rental), INTERVAL 6 MONTH)`